<h1>🍃 Leech Bot — Direct Colab Runner (Testing Only)</h1>

- 🔗 **Repo:** https://github.com/awaradeveloper/leech.git
- ⚠️ Runs natively inside the Colab VM (no Docker). Torrent (qBittorrent) and Usenet (SABnzbd) support are **disabled** by default since their daemons aren't set up here — only direct links, yt-dlp, and Mega/Drive/Rclone paths work out of the box.
- ⚠️ Session dies when Colab disconnects (idle timeout or ~12h cap). Not for real uptime — use Heroku/VPS/Docker for that.

Run the cells in order.

In [ ]:
#@title <center><h3>1. Clone the Repo</h3></center><br>

#@markdown ---
Repo_URL = "https://github.com/awaradeveloper/leech.git" #@param {type:"string"}
Branch = "main" #@param {type:"string"}
#@markdown ---

import os

if os.path.isdir("leech"):
    !rm -rf leech

!git clone --branch {Branch} {Repo_URL} leech
%cd leech
print("Repo cloned into /content/leech")

In [ ]:
#@title <center><h3>2. Install System Dependencies</h3></center><br>

#@markdown Installs aria2 (direct-link/yt-dlp downloads), ffmpeg, and mediainfo. Skips qBittorrent/SABnzbd since those daemons aren't wired up in this notebook.

!apt-get update -qq
!apt-get install -y -qq aria2 ffmpeg mediainfo > /dev/null
print("System packages installed: aria2, ffmpeg, mediainfo")

In [ ]:
#@title <center><h3>3. Install Python Requirements</h3></center><br>

!pip install -q -r requirements.txt
print("Python dependencies installed.")

In [ ]:
#@title <center><h3>4. Configure Bot (config.py)</h3></center><br>

#@markdown #### ***Fill these Variables*** **(All are Mandatory)**
BOT_TOKEN = ""  # @param {type:"string"}
TELEGRAM_API = 0  # @param {type:"integer"}
TELEGRAM_HASH = ""  # @param {type:"string"}
OWNER_ID = 0  # @param {type:"integer"}
DATABASE_URL = ""  # @param {type:"string"}

#@markdown ---
#@markdown ### ***Optional***
UPSTREAM_REPO = "https://github.com/awaradeveloper/leech.git"  # @param {type:"string"}
UPSTREAM_BRANCH = "main"  # @param {type:"string"}
BASE_URL = ""  # @param {type:"string"}
TIMEZONE = "Asia/Kolkata"  # @param {type:"string"}

#@markdown ---
#@markdown ### ***Feature toggles (leave as-is unless you set up qBittorrent/SABnzbd/JDownloader yourself)***
DISABLE_TORRENTS = True  # @param {type:"boolean"}
DISABLE_NZB = True  # @param {type:"boolean"}
DISABLE_JD = True  # @param {type:"boolean"}

#@markdown ---

if not all([BOT_TOKEN, TELEGRAM_API, TELEGRAM_HASH, OWNER_ID, DATABASE_URL]):
    raise ValueError("Please fill in all Mandatory Variables.")

config_lines = [
    f'BOT_TOKEN = "{BOT_TOKEN}"',
    f'TELEGRAM_API = {TELEGRAM_API}',
    f'TELEGRAM_HASH = "{TELEGRAM_HASH}"',
    f'OWNER_ID = {OWNER_ID}',
    f'DATABASE_URL = "{DATABASE_URL}"',
    f'UPSTREAM_REPO = "{UPSTREAM_REPO}"',
    f'UPSTREAM_BRANCH = "{UPSTREAM_BRANCH}"',
    f'BASE_URL = "{BASE_URL}"',
    f'TIMEZONE = "{TIMEZONE}"',
    f'DISABLE_TORRENTS = {DISABLE_TORRENTS}',
    f'DISABLE_NZB = {DISABLE_NZB}',
    f'DISABLE_JD = {DISABLE_JD}',
]

with open("config.py", "w") as f:
    f.write("\n".join(config_lines) + "\n")

print("config.py written:")
print("\n".join(config_lines))

In [ ]:
#@title <center><h3>5. Start the aria2c Daemon</h3></center><br>

#@markdown Starts aria2c in RPC daemon mode, matching what `setpkgs.sh` expects (torrents/SABnzbd are skipped since ARIA2C is the only binary we have here).

!bash setpkgs.sh $(which aria2c) "" 20 ""
print("aria2c daemon started.")

In [ ]:
#@title <center><h3>6. Run the Bot</h3></center><br>

#@markdown This cell runs in the foreground and keeps the bot alive as long as this cell is running. Stop the cell (■) to stop the bot.

!python update.py
!python -m bot

In [ ]:
#@title <center><h3>(Optional) Tail Logs in a Separate Cell</h3></center><br>

#@markdown If you started the bot in the background instead (e.g. by changing cell 6 to `!nohup python -m bot > bot.log 2>&1 &`), use this cell to watch the log file.

!tail -n 100 -f log.txt